# 00 · 검증 (도구 점검 + 구현 검증)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/00_check.ipynb)

이후 모든 블록 구현이 통과해야 하는 **판정 기준**을 먼저 점검하는 노트북입니다.

1. 환경 준비
2. 출력 비교 도구 자체 점검: 같은 함수면 PASS, 다르면 FAIL이 나오는지
3. 자체 일관성 검사 (causal mask 누수)
4. HF 원본 구조 읽기 (가중치 다운로드 없이 헤더만)
5. 구조(shape) 대조 도구 점검
6. 단계별 검증 현황표
7. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요.

## 1. 환경 준비

In [1]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip -q install -U -r requirements-colab.txt
# Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
!pip -q uninstall -y torchao

/content/tmp_llm_dit
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 59.6 MB/s eta 0:00:00


In [2]:
import sys, torch, transformers, diffusers
print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("transformers", transformers.__version__)
print("diffusers   ", diffusers.__version__)
!git log -1 --format='repo commit: %h %s'

python       3.13.15
torch        2.11.0+cu130 | CUDA: True | Tesla T4
transformers 5.19.0
diffusers    0.41.0
repo commit: 5eada1c Rename verify notebook to 00_check (avoid cached broken copy)


In [3]:
# gated 모델(Krea-2 등) 헤더를 읽을 때만 필요. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN 설정됨")
except Exception:
    print("HF_TOKEN 없음 (공개 모델만 확인 가능)")

HF_TOKEN 설정됨


## 2. 출력 비교 도구 자체 점검

원본 예시로 HF GPT-2를 씁니다.
- **같은 구조**인 별도 모델에 가중치를 복사하면 → ✅ PASS가 나와야 합니다.
- **일부러 수식을 바꾼** 모델(LayerNorm eps 변경)은 → ❌ FAIL이 나와야 합니다.
- `hook_pairs`로 층별 중간 출력도 비교해서, 어느 층부터 어긋나는지 보여줍니다.

In [4]:
from transformers import GPT2Config, GPT2Model
from miniarch import verify as V

V.set_deterministic(0)
ref, inputs = V.cases.ref_gpt2()

same = GPT2Model(ref.config)                                  # 같은 구조, 다른 랜덤 초기화
cfg_bad = GPT2Config(**{**ref.config.to_dict(), "layer_norm_epsilon": 1e-1})
bad = GPT2Model(cfg_bad)                                      # 수식이 살짝 다름

layers = [("h.0", "h.0"), ("h.1", "h.1")]
print("[같은 구조]");    print(V.check_equivalence(ref, same, inputs, hook_pairs=layers).summary())
print("\n[다른 수식]"); print(V.check_equivalence(ref, bad,  inputs, hook_pairs=layers).summary())

[같은 구조]
✅ PASS  (복사 28개)
  ✅ h.0  <->  h.0                            max_abs=0.00e+00 
  ✅ h.1  <->  h.1                            max_abs=0.00e+00 
  ✅ [final output]                           max_abs=0.00e+00 

[다른 수식]
❌ FAIL  (복사 28개)
  ❌ h.0  <->  h.0                            max_abs=5.48e-02 
  ❌ h.1  <->  h.1                            max_abs=7.47e-02 
  ❌ [final output]                           max_abs=3.32e+00 


## 3. 자체 일관성 검사: causal mask 누수

뒤쪽 토큰을 바꿨을 때 앞쪽 출력이 변하면 미래 정보가 새고 있다는 뜻입니다.
HF Llama(정상)는 PASS, mask를 빼고 양방향으로 본 경우는 FAIL이 나와야 합니다.

In [5]:
dev = "cuda" if torch.cuda.is_available() else "cpu"
llama, _ = V.cases.ref_llama()
llama = llama.to(dev).eval(); V.force_eager(llama)

causal = lambda x: llama(input_ids=x).last_hidden_state
# 모든 위치를 볼 수 있는 4D mask(0 = 허용)를 넘겨 일부러 양방향으로 만든 경우
bidir  = lambda x: llama(input_ids=x, attention_mask=torch.zeros(x.shape[0], 1, x.shape[1], x.shape[1], device=x.device)).last_hidden_state

for name, f in [("Llama causal", causal), ("Llama 양방향(누수)", bidir)]:
    try:
        r = V.causal_leak_test(f, vocab_size=V.cases.VOCAB, device=dev)
        print(f"{'✅' if r.ok else '❌'} {name:<18} max_abs={r.max_abs:.2e}")
    except Exception as e:
        print(f"⚠️ {name}: {type(e).__name__}: {e}")

✅ Llama causal       max_abs=0.00e+00
❌ Llama 양방향(누수)      max_abs=1.37e+00


## 4. HF 원본 구조 읽기 (헤더만, 다운로드 없음)

safetensors 헤더를 HTTP range 요청으로 읽어서 실제 가중치 이름, shape, dtype을 확인합니다.
routed expert는 개수가 많아 숨깁니다.

In [6]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        h = {k: v for k, v in h.items() if ".experts." not in k}
        V.show(h, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek("deepseek-ai/DeepSeek-V4-Flash", r"layers\.0\.")              # MLA(V4) + mHC + sparse
peek("Qwen/Qwen3.6-27B",             r"layers\.(0|3)\.")          # Gated DeltaNet(0) / Gated Attn(3)
peek("moonshotai/Kimi-K3",           r"layers\.(1|4)\.")          # KDA(1) / MLA(4)
peek("Qwen/Qwen-Image-2.1",          r"blocks\.0\.|transformer_blocks\.0\.", subfolder="transformer")


===== deepseek-ai/DeepSeek-V4-Flash  /layers\.0\./
{'model_type': 'deepseek_v4', 'hidden_size': 4096, 'num_hidden_layers': 43, 'num_attention_heads': 64, 'num_key_value_heads': 1, 'head_dim': 512}
layers.0.attn.attn_sink                                                F32      [64]
layers.0.attn.kv_norm.weight                                           BF16     [512]
layers.0.attn.q_norm.weight                                            BF16     [1024]
layers.0.attn.wkv.scale                                                F8_E8M0  [4, 32]
layers.0.attn.wkv.weight                                               F8_E4M3  [512, 4096]
layers.0.attn.wo_a.scale                                               F8_E8M0  [64, 32]
layers.0.attn.wo_a.weight                                              F8_E4M3  [8192, 4096]
layers.0.attn.wo_b.scale                                               F8_E8M0  [32, 64]
layers.0.attn.wo_b.weight                                              F8_E4M3  [4096, 8192]


## 5. 구조(shape) 대조 도구 점검

원본 config로 **meta device**(메모리 사용 0)에 모델을 만들고, 그 state_dict를 원본 헤더와 대조합니다.
여기서는 HF의 Qwen3 구현과 실제 Qwen3-0.6B 체크포인트를 비교하니 ✅가 나와야 정상입니다.
나중에는 HF 구현 자리에 **내 구현**을 넣어서 같은 방식으로 검사합니다.

In [7]:
from transformers import AutoConfig, AutoModelForCausalLM

repo = "Qwen/Qwen3-0.6B"
cfg = AutoConfig.from_pretrained(repo)
mine_shapes = V.meta_state_shapes(lambda: AutoModelForCausalLM.from_config(cfg))
orig = V.read_model_header(repo)
print(V.compare_shapes(mine_shapes, orig, scope=r"layers\.0\.|embed_tokens|model\.norm").summary())

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

✅ PASS  일치 13개


## 6. 단계별 검증 현황표

- 원본 builder가 현재 라이브러리 버전에서 동작하는지 먼저 확인합니다.
- 그다음 전체 표를 출력합니다. 지금은 내 구현이 없어서 ⬜ 상태가 정상이고, 구현을 추가할 때마다 ✅로 바뀝니다.

In [8]:
V.cases.smoke_refs()

✅ A0   gpt2     params=112,384  out=(2, 16, 64)
✅ A1   Llama    params=82,240  out=(2, 16, 64)
✅ B0   DiT      params=202,448  out=(2, 1, 28, 28)


In [9]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
status = V.cases.run_all()
status[["stage", "block", "기준", "status"]]

,stage,block,기준,status
0,A0,"GPT-2 (MHA, learned pos, LayerNorm, GELU)",gpt2,❌ FAIL
1,A1,"Llama (RoPE, RMSNorm, SwiGLU, GQA)",Llama,✅ PASS
2,A2,QK-norm / SWA+full / sink / partial RoPE,MiMo-V2.6,⬜ 원본 builder 미작성
3,A3,"MoE (fine-grained + shared, aux-free)",DeepSeek-V4 · GLM-5.3,⬜ 원본 builder 미작성
4,A4,"MLA (V3 → V4: 1 KV head, o_lora)",DeepSeek-V4,⬜ 원본 builder 미작성
5,A5a,DSA indexer (+층 간 공유),GLM-5.3,⬜ 원본 builder 미작성
6,A5b,블록 top-k sparse (MSA),MiniMax-M3,⬜ 원본 builder 미작성
7,A6a,Gated DeltaNet + Gated Attention,Qwen3.6/3.8-27B,⬜ 원본 builder 미작성
8,A6b,KDA + gated MLA,Kimi-K3,⬜ 원본 builder 미작성
9,A7a,Hyper-Connections / mHC,DeepSeek-V4 · Qwen3.8-Flash-Next,⬜ 원본 builder 미작성


In [10]:
# 검사한 블록의 층별 상세 (PASS 도 어느 층까지 오차가 얼마인지 확인)
for _, r in status[~status.status.str.startswith("⬜")].iterrows():
    print(f"--- {r.stage} {r.block}  {r.status}\n{r.detail}\n")

--- A0 GPT-2 (MHA, learned pos, LayerNorm, GELU)  ❌ FAIL
❌ FAIL  (복사 26개)
  복사 안 된 mine 파라미터 2: ['blocks.0.attn.qkv.bias', 'blocks.1.attn.qkv.bias']
  ✅ h.0  <->  blocks.0                       max_abs=7.45e-09 
  ✅ h.1  <->  blocks.1                       max_abs=1.55e-08 
  ✅ [final output]                           max_abs=4.77e-07 

--- A1 Llama (RoPE, RMSNorm, SwiGLU, GQA)  ✅ PASS
✅ PASS  (복사 20개)
  ✅ layers.0  <->  layers.0                  max_abs=0.00e+00 
  ✅ layers.1  <->  layers.1                  max_abs=0.00e+00 
  ✅ [final output]                           max_abs=0.00e+00 

--- B0 DiT (adaLN-Zero)  ✅ PASS
✅ PASS  (복사 39개)
  ✅ pos_embed  <->  patch_embed              max_abs=0.00e+00 
  ✅ transformer_blocks.0  <->  blocks.0      max_abs=2.38e-07 
  ✅ transformer_blocks.1  <->  blocks.1      max_abs=4.77e-07 
  ✅ [final output]                           max_abs=2.98e-07 



### 6-1. 내 구현 자체 점검

- **causal 누수**: GPT-2, Llama 구현이 미래 토큰을 보지 않는지 확인합니다.
- **attention map 캡처**: 모든 블록의 `attn_probs` 지점에서 확률 텐서를 hook으로 꺼낼 수 있는지 확인합니다. 각 행의 합은 1이어야 합니다. 이 지점이 이후 attention map 분석에 쓰입니다.

In [11]:
from miniarch.models.gpt2 import GPT2, GPT2Config
from miniarch.models.llama import Llama, LlamaConfig
from miniarch.models.dit import DiT, DiTConfig

V.set_deterministic(0)
lms = {"GPT2": GPT2(GPT2Config(vocab_size=128)).to(dev).eval(),
       "Llama": Llama(LlamaConfig(vocab_size=128)).to(dev).eval()}
for name, m in lms.items():
    r = V.causal_leak_test(lambda x, m=m: m(x).last_hidden_state, vocab_size=128, device=dev)
    print(f"{'✅' if r.ok else '❌'} {name:<6} causal leak  max_abs={r.max_abs:.2e}")

def probe_attn(model, call, causal):
    names = [n for n, _ in model.named_modules() if n.endswith("attn_probs")]
    with V.capture(model, names) as st, torch.no_grad():
        call()
    p = st[names[0]]
    msg = f"{type(model).__name__:<6} {len(names)}개 층 캡처, shape={tuple(p.shape)}, 행 합 오차={(p.sum(-1) - 1).abs().max().item():.1e}"
    if causal:
        msg += f", 미래 토큰에 준 확률 최대={p.triu(1).abs().max().item():.1e}"
    print(msg)

x = torch.randint(0, 128, (2, 16), device=dev)
for m in lms.values():
    probe_attn(m, lambda m=m: m(x), causal=True)
dit = DiT(DiTConfig()).to(dev).eval()
img = torch.randn(2, 1, 28, 28, device=dev)
probe_attn(dit, lambda: dit(img, torch.tensor([10, 500], device=dev), torch.tensor([3, 7], device=dev)), causal=False)

✅ GPT2   causal leak  max_abs=0.00e+00
✅ Llama  causal leak  max_abs=0.00e+00
GPT2   4개 층 캡처, shape=(2, 4, 16, 16), 행 합 오차=1.2e-07, 미래 토큰에 준 확률 최대=0.0e+00
Llama  4개 층 캡처, shape=(2, 4, 16, 16), 행 합 오차=1.2e-07, 미래 토큰에 준 확률 최대=0.0e+00
DiT    6개 층 캡처, shape=(2, 4, 49, 49), 행 합 오차=1.8e-07


## 7. 결과 저장

- `results/`에 현황표를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장**으로 이 저장소의 `notebooks/`에 저장하세요.
- Drive에도 남기려면 아래 셀의 `SAVE_TO_DRIVE = True`로 바꾸세요.

In [12]:
import datetime, shutil
stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
os.makedirs("results", exist_ok=True)
path = f"results/00_check_{stamp}.csv"
status.to_csv(path, index=False)
print("saved:", path)

SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = "/content/drive/MyDrive/tmp_llm_dit/results"
    os.makedirs(dst, exist_ok=True)
    shutil.copy(path, dst)
    print("drive:", dst)

saved: results/00_check_20261014-0552.csv
